# Chapter 12 — Transformer Block from Scratch

Runnable companion to `docs/11_transformer_intro.md`. We build a
**single Transformer encoder block** with bare PyTorch
(`nn.MultiheadAttention` plus a small FFN), wrap it in residual +
LayerNorm, and stack `N` of them.

Everything is shape-checked at every step. Generated by
`src/build_chapter_11_transformer_intro.py`.

## Setup

In [1]:
import math

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

torch.manual_seed(0); np.random.seed(0)
DEVICE = torch.device("cpu")
print("torch", torch.__version__, "device", DEVICE)

torch 2.12.0+cu130 device cpu


## 1. Positional encoding

Attention is permutation-equivariant: it cannot tell which token came
first. We add a deterministic positional pattern to the embeddings so
each position has a unique fingerprint.

The original paper uses sinusoids of geometrically-spaced frequencies:

In [2]:
def sinusoidal_positional_encoding(max_len: int, d_model: int) -> torch.Tensor:
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(max_len, dtype=torch.float).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float)
                         * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe


pe = sinusoidal_positional_encoding(max_len=80, d_model=64)
print("PE shape:", tuple(pe.shape))

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(pe.numpy(), aspect="auto", cmap="RdBu", vmin=-1, vmax=1)
ax.set_xlabel("feature dimension"); ax.set_ylabel("position")
ax.set_title("Sinusoidal positional encoding")
plt.colorbar(im, ax=ax, fraction=0.046)
plt.tight_layout(); plt.show()

PE shape: (80, 64)


/tmp/ipykernel_355254/3951376262.py:19: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  plt.tight_layout(); plt.show()


Each *row* is the positional fingerprint of a token at that position.
The lower-frequency columns vary slowly across positions and the
higher-frequency columns vary quickly — together they let the network
disambiguate up to `max_len` distinct positions.

## 2. A complete encoder block

Pre-norm style (modern default — much more stable at depth):

```
y_1 = x   + MultiHeadSelfAttention(LayerNorm(x))
y_2 = y_1 + FFN(LayerNorm(y_1))
```

In [3]:
class EncoderBlock(nn.Module):
    def __init__(self, d_model=64, num_heads=4, d_ff=256, dropout=0.0):
        super().__init__()
        self.ln1  = nn.LayerNorm(d_model)
        self.attn = nn.MultiheadAttention(d_model, num_heads,
                                          dropout=dropout, batch_first=True)
        self.ln2  = nn.LayerNorm(d_model)
        self.ffn  = nn.Sequential(
            nn.Linear(d_model, d_ff),
            nn.GELU(),
            nn.Linear(d_ff, d_model),
        )
        self.drop = nn.Dropout(dropout)

    def forward(self, x, attn_mask=None, key_padding_mask=None):
        x_ln = self.ln1(x)
        a, _ = self.attn(x_ln, x_ln, x_ln,
                         attn_mask=attn_mask,
                         key_padding_mask=key_padding_mask,
                         need_weights=False)
        x = x + self.drop(a)
        x = x + self.drop(self.ffn(self.ln2(x)))
        return x


block = EncoderBlock(d_model=64, num_heads=4, d_ff=256)
x = torch.randn(2, 10, 64)             # (B, T, d_model)
y = block(x)
print(f"input  shape: {tuple(x.shape)}")
print(f"output shape: {tuple(y.shape)}")
print(f"block params : {sum(p.numel() for p in block.parameters()):,}")

input  shape: (2, 10, 64)
output shape: (2, 10, 64)
block params : 49,984


The output shape **must** equal the input shape — that is what makes
the residual `+` and stacking work.

## 3. A stack of N blocks

In [4]:
class TinyTransformerEncoder(nn.Module):
    def __init__(self, vocab_size, d_model=64, num_heads=4, d_ff=256,
                 num_layers=4, max_len=128, num_classes=2, dropout=0.0):
        super().__init__()
        self.embed = nn.Embedding(vocab_size, d_model)
        self.register_buffer(
            "pe", sinusoidal_positional_encoding(max_len, d_model), persistent=False
        )
        self.blocks = nn.ModuleList([
            EncoderBlock(d_model, num_heads, d_ff, dropout)
            for _ in range(num_layers)
        ])
        self.ln_f = nn.LayerNorm(d_model)
        self.head = nn.Linear(d_model, num_classes)

    def forward(self, token_ids):
        T = token_ids.size(1)
        x = self.embed(token_ids) + self.pe[:T]
        for block in self.blocks:
            x = block(x)
        x = self.ln_f(x).mean(dim=1)        # mean-pool over the sequence
        return self.head(x)


model = TinyTransformerEncoder(vocab_size=128, d_model=64, num_heads=4,
                               d_ff=256, num_layers=4, num_classes=2)
ids = torch.randint(0, 128, (2, 32))
logits = model(ids)
n_params = sum(p.numel() for p in model.parameters())
print(f"input  : {tuple(ids.shape)}")
print(f"logits : {tuple(logits.shape)}")
print(f"params : {n_params:,}  (4-block, d_model=64, num_heads=4)")

input  : (2, 32)
logits : (2, 2)
params : 208,386  (4-block, d_model=64, num_heads=4)


## 4. Parameter count vs. depth and width

A tiny ablation: how does the parameter count scale as we change
`num_layers` and `d_model`? (No training — just constructor inspection.)

In [5]:
def count_params(d_model, num_heads, d_ff, num_layers):
    m = TinyTransformerEncoder(
        vocab_size=128, d_model=d_model, num_heads=num_heads,
        d_ff=d_ff, num_layers=num_layers, num_classes=2,
    )
    return sum(p.numel() for p in m.parameters())


print(f"{'config':<40s} {'params':>12s}")
for L in [1, 2, 4, 8]:
    n = count_params(64, 4, 256, L)
    print(f"  L={L}, d_model=64, d_ff=256  ".ljust(40) + f"{n:>12,}")
print()
for d in [32, 64, 128, 256]:
    n = count_params(d, 4, d * 4, 4)
    print(f"  d_model={d}, d_ff={d*4}, L=4  ".ljust(40) + f"{n:>12,}")

config                                         params
  L=1, d_model=64, d_ff=256                   58,434
  L=2, d_model=64, d_ff=256                  108,418
  L=4, d_model=64, d_ff=256                  208,386
  L=8, d_model=64, d_ff=256                  408,322

  d_model=32, d_ff=128, L=4                   55,042
  d_model=64, d_ff=256, L=4                  208,386
  d_model=128, d_ff=512, L=4                 809,986
  d_model=256, d_ff=1024, L=4              3,192,834


Parameter count grows roughly **linearly with `num_layers`** and
**quadratically with `d_model`** (because both attention QKV and the FFN
inner dimension scale with `d_model`). This is why scaling-law papers
trade depth-vs-width carefully — depth is cheap, width is expensive.

## 5. Quick check

- Why pre-norm and not post-norm?
  Pre-norm `x + sublayer(LN(x))` puts the residual stream on a clean
  gradient highway and trains stably at depth without warmup tricks.
- Why is `d_ff = 4 * d_model` so common?
  This ratio comes from the original paper and has stuck because the
  FFN is where "real computation" happens — wider helps; 4× is the sweet
  spot empirically.
- Could we replace `mean(dim=1)` with the `[CLS]` trick?
  Yes — prepend a learned `[CLS]` token to every input and use *its*
  final hidden state for classification. That is exactly what BERT does.